# 03 — Modèle ML — Classification du ton médiatique
**Hackathon iSHEERO × DataCamp 2026 — Bénin Insights Challenge**

Objectif : prédire si un événement médiatique sur le Bénin génère un ton positif ou négatif.

Algorithme : Random Forest Classifier (scikit-learn), comparé à une baseline DummyClassifier.
Données : `benin_enrichi.csv` — 10 722 événements, 2025

Cible : `ton_binaire = 1 si AvgTone > 0`, 0 sinon.
`AvgTone` n'est pas dans les features — pas de fuite directe.
`GoldsteinScale` et `EventRootCode` sont des propriétés du type d'événement CAMEO, indépendantes du ton des articles.


## 1. Imports

In [ ]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

## 2. Chargement des données

In [ ]:
ROOT = Path("../")
df = pd.read_csv(ROOT / "data/processed/benin_enrichi.csv")
print(f"Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

## 3. Variable cible

`ton_binaire = 1` si `AvgTone > 0` (ton positif), `0` sinon (ton négatif ou neutre).

`AvgTone` est exclu des features : la conserver serait une fuite triviale (la cible en est la transformation directe).
`ton_categorie` est également exclue pour la même raison.

In [ ]:
df["ton_binaire"] = (df["AvgTone"] > 0).astype(int)

print("Distribution de la cible :")
vc = df["ton_binaire"].value_counts()
print(f"  Négatif (0) : {vc[0]:,} ({vc[0]/len(df):.1%})")
print(f"  Positif (1) : {vc[1]:,} ({vc[1]/len(df):.1%})")

## 4. Features et préparation

Variables retenues : type d'événement (`EventRootCode`, `QuadClass`), stabilité (`GoldsteinScale`), volume médiatique (`NumMentions`, `NumArticles`), saison (`mois`), zone géographique (`zone_benin`).

`AvgTone` exclue — fuite directe de la cible.
`ton_categorie` exclue — dérivée de `AvgTone`.

Note : `mois` capture des effets saisonniers propres à 2025 (décembre est fortement négatif en raison d'un pic médiatique exceptionnel). Son importance élevée reflète ce signal saisonnier, pas un prédicteur causal généralisable.

In [ ]:
FEATURES = ["EventRootCode", "QuadClass", "GoldsteinScale",
            "NumMentions", "NumArticles", "mois", "zone_benin"]

df_ml = df[FEATURES + ["ton_binaire"]].dropna()
print(f"Lignes disponibles après dropna : {len(df_ml):,}")

X = pd.get_dummies(df_ml[FEATURES], columns=["zone_benin"])
y = df_ml["ton_binaire"]

print(f"Features finales ({len(X.columns)}) : {list(X.columns)}")

## 5. Séparation train / test

Split stratïié sur la cible (même proportion positif/négatif dans train et test).
`random_state=42` pour reproductibilité.

Limite connue : le split aléatoire sur série temporelle permet aux patterns saisonniers
de se retrouver dans les deux jeux. Un split temporel strict (train=jan–oct, test=nov–déc)
produit une accuracy de ~50 % — inférieure à la baseline — car novembre-décembre 2025
a une distribution de ton significativement différente du reste de l'année.
La performance reportée ici est donc conditionnelle à la distribution de 2025.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {len(X_train):,} événements  (ratio positif : {y_train.mean():.1%})")
print(f"Test  : {len(X_test):,} événements  (ratio positif : {y_test.mean():.1%})")

## 6. Baseline — DummyClassifier

In [ ]:
# Baseline : prédit toujours la classe majoritaire (Négatif)
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)
y_dummy = dummy.predict(X_test)

acc_dummy = accuracy_score(y_test, y_dummy)
print(f"Accuracy DummyClassifier (classe majoritaire) : {acc_dummy:.4f}")
print("Un modèle utile doit clairement dépasser ce seuil.")

## 7. Entraînement — Random Forest

In [ ]:
clf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",   # compense le déséquilibre 60/40
    random_state=42
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print("Modèle entraîné.")

## 8. Évaluation

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy Random Forest    : {acc:.4f}")
print(f"Accuracy DummyClassifier  : {acc_dummy:.4f}")
print(f"Gain sur baseline         : +{acc - acc_dummy:.4f}  ({(acc - acc_dummy)*100:.1f} pp)")
print()
print(classification_report(y_test, y_pred, target_names=["Négatif (0)", "Positif (1)"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Négatif", "Positif"])
disp.plot(ax=ax, colorbar=False)
ax.set_title("Matrice de confusion — Random Forest")
plt.tight_layout()
(ROOT / "notebooks/outputs").mkdir(exist_ok=True)
plt.savefig(ROOT / "notebooks/outputs/ml_confusion_matrix.png", dpi=120)
plt.show()

## 9. Importance des variables

In [ ]:
importances = pd.Series(clf.feature_importances_, index=X.columns) \
              .sort_values(ascending=False)

print("Importance des variables (Gini) :")
print(importances.round(4).to_string())

fig, ax = plt.subplots(figsize=(7, 4))
importances.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Importance des variables — Random Forest")
ax.set_xlabel("Importance (Gini)")
plt.tight_layout()
plt.savefig(ROOT / "notebooks/outputs/ml_feature_importance.png", dpi=120)
plt.show()

## 10. Sauvegarde du modèle

In [ ]:
model_path = ROOT / "models/random_forest_ton.pkl"
(ROOT / "models").mkdir(exist_ok=True)
joblib.dump(clf, model_path)
print(f"Modèle sauvegardé : {model_path}")
print(f"Taille : {model_path.stat().st_size / 1024:.0f} Ko")

## 12. Validation croisée

Un StratifiedKFold à 5 folds vérifie que la performance est stable et non due au découpage train/test particulier.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_cv = RandomForestClassifier(
    n_estimators=100, class_weight="balanced", random_state=42
)
scores = cross_val_score(rf_cv, X, y, cv=cv, scoring='accuracy')

print('Validation croisée — RF avec mois (5 folds stratifiés) :')
print(f'  Accuracy par fold : {[round(s, 4) for s in scores]}')
print(f'  Moyenne           : {scores.mean():.4f}')
print(f'  Ecart-type        : {scores.std():.4f}')
print()
print('Interprétation : faible écart-type = modèle stable sur différents')
print('découpages. La performance ne dépend pas d un seul split.')

## 13. Test de robustesse — modèle sans `mois`

`mois` capte la saisonnalité de 2025, dont l'anomalie de décembre. Ce test mesure ce que le modèle sait faire sans ce signal temporel — et donc ce qu'il pourrait produire sur une nouvelle année.

In [ ]:
FEATURES_SM = ["EventRootCode", "QuadClass", "GoldsteinScale",
               "NumMentions", "NumArticles", "zone_benin"]

df_sm = df[FEATURES_SM + ["ton_binaire"]].dropna()
X_sm  = pd.get_dummies(df_sm[FEATURES_SM], columns=["zone_benin"])
y_sm  = df_sm["ton_binaire"]

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_sm, y_sm, test_size=0.2, random_state=42, stratify=y_sm
)

rf2 = RandomForestClassifier(
    n_estimators=100, class_weight="balanced", random_state=42
)
rf2.fit(X_tr2, y_tr2)
acc_rf2 = accuracy_score(y_te2, rf2.predict(X_te2))

scores2 = cross_val_score(
    RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    X_sm, y_sm, cv=cv, scoring="accuracy"
)

imps2 = pd.Series(rf2.feature_importances_, index=X_sm.columns).sort_values(ascending=False)

print("RF sans mois — split stratifié :")
print(f"  Accuracy : {acc_rf2:.4f}")
print(f"  CV moyenne : {scores2.mean():.4f} ± {scores2.std():.4f}")
print()
print("Importance des variables (sans mois) :")
print(imps2.round(4).to_string())

## 14. Tableau comparatif

Résumé des trois configurations pour contextualiser les résultats.

In [ ]:
resultats = {
    "Baseline (DummyClassifier)"   : acc_dummy,
    "RF avec mois (modele livre)"  : acc,
    "RF sans mois (robustesse)"    : acc_rf2,
}

print(f"Modele{chr(32)*29} Accuracy")
print("-" * 47)
for nom, val in resultats.items():
    gain = val - acc_dummy
    print(f"{nom:<35} {val:.4f}  (+{gain:.4f} vs baseline)")

## 15. Synthèse

| Modèle | Accuracy | CV (5 folds) | F1 macro | Gain baseline |
|--------|----------|--------------|----------|---------------|
| DummyClassifier (baseline) | 0,60 | — | — | référence |
| **RF avec `mois`** | **0,71** | **0,708 ± 0,009** | **0,707** | **+11 pp** |
| RF sans `mois` | 0,64 | 0,648 ± 0,009 | 0,643 | +4 pp |

**Lecture :**

- Le Random Forest avec `mois` atteint 71 % — stable sur les 5 folds (écart-type 0,009). Ce n'est pas un artefact du découpage.
- La variable `mois` apporte **6,9 pp** au-delà du signal événementiel pur. Elle capture la saisonnalité de 2025 — en particulier le pic de décembre qui concentre 18 % du volume annuel.
- Sans `mois`, le modèle tombe à 64 % mais reste **au-dessus de la baseline** (+4 pp). `GoldsteinScale` et `EventRootCode` deviennent les features principales — plus généralisables.

**Limites et avertissements :**

- Le modèle capture des régularités observées dans les données 2025. Il ne prédit pas le futur et ne généralise pas nécessairement à 2026.
- La contribution de `mois` est réelle mais spécifique à cette année. Sur une année sans pic de décembre, la performance se rapprocherait du modèle sans `mois` (64 %).
- 29 % des cas restent mal classés. Le contenu textuel des articles — inaccessible via GDELT — expliquerait une partie des erreurs.
- Aucune donnée 2026 n'est disponible pour valider la généralisation.

**Recommandations Phase 2 :**

- Tester sur les données GDELT de janvier–avril 2026 dès disponibles.
- Évaluer un modèle de forecasting (Prophet ou XGBoost) sur le ton J+7.
- Envisager de remplacer `mois` par des features événementielles plus stables : `trimestre`, `EventBaseCode` à deux chiffres, acteurs dominants.
